# Nama: Taqiyuddin Ja'far
## NIM: 250401020186
### Kelas: IF405

## Langkah 1: Generate & Eksplorasi Dataset Transaksi

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

# Buat dataset sintetis
np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
  n_item = np.random.randint(2, 6)
  transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
  if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
    transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


### Catatan
- Dataset ini terdiri dari 50 transaksi, di mana setiap transaksi berisi antara 2 hingga 5 produk yang dipilih secara acak dari daftar `produk`.
- Selain itu, untuk 20 transaksi pertama, kita sengaja menyuntikkan pola agar 'Roti' sering muncul bersama 'Selai' untuk menguji algoritma Apriori nantinya.
- Output menunjukkan 3 contoh transaksi pertama dan total jumlah transaksi yang dibuat (50 transaksi). Ini mengkonfirmasi bahwa dataset telah berhasil dibuat dan memiliki struktur yang diharapkan.

## Langkah 2: One-Hot Encoding Transaksi

In [2]:
from mlxtend.preprocessing import TransactionEncoder

# Ubah daftar transaksi menjadi tabel one-hot encoding menggunakan TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


### Catatan

Menggunakan `TransactionEncoder` dari pustaka `mlxtend` untuk mengubah daftar transaksi mentah menjadi format _*one-hot encoding*_. Dalam format ini, setiap baris mewakili satu transaksi dan setiap kolom mewakili satu produk. Nilai `True` menunjukkan bahwa produk tersebut ada dalam transaksi, sedangkan `False` berarti produk tidak ada.

Output `df.head()` menampilkan 5 baris pertama dari DataFrame hasil _one-hot encoding_. Ini memperjelas bagaimana setiap transaksi direpresentasikan sebagai kombinasi produk yang ada (True) atau tidak ada (False).

## Langkah 3: Cari Frequent Itemset dengan Apriori

In [5]:
from mlxtend.frequent_patterns import apriori
import warnings

# Temporarily ignore DeprecationWarnings from jupyter_client to clean up output
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

for ms in [0.05, 0.1, 0.2]:
  freq = apriori(df, min_support=ms, use_colnames=True)
  print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Teh, Selai)


### Catatan

Pada langkah ini, kita mengaplikasikan algoritma Apriori untuk menemukan _frequent itemset_ (kumpulan produk yang sering muncul bersamaan) dari data transaksi yang telah di-_one-hot encoded_. Kita menguji tiga nilai `min_support` yang berbeda (0.05, 0.1, 0.2) untuk melihat bagaimana ambang batas ini memengaruhi jumlah _itemset_ yang ditemukan.

Output menunjukkan:
- `min_support=0.05`: ditemukan 74 _itemset_.
- `min_support=0.1`: ditemukan 44 _itemset_.
- `min_support=0.2`: ditemukan 13 _itemset_.

Kita kemudian memilih `min_support=0.1` karena menghasilkan jumlah _itemset_ yang wajar untuk dianalisis (tidak terlalu banyak dan tidak kosong). Tabel `freq_items.head(10)` menampilkan 10 _itemset_ teratas yang diurutkan berdasarkan nilai _support_ tertinggi. _Support_ mengukur seberapa sering _itemset_ tersebut muncul dalam dataset transaksi. Misalnya, '(Selai)' memiliki _support_ 0.52, yang berarti Selai muncul di 52% dari total transaksi.

## Langkah 4: Bentuk & Saring Aturan Asosiasi

In [8]:
from mlxtend.frequent_patterns import association_rules

# Bentuk aturan asosiasi, saring dengan min_confidence dan min_lift
rules = association_rules(freq_items, metric='confidence',min_threshold=0.5)

# urutkan berdasarkan Lift tertinggi
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents',
             'support', 'confidence', 'lift']].head(10))

# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

         antecedents consequents  support  confidence      lift
8        (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
15  (Mentega, Selai)      (Kopi)     0.10    0.625000  1.953125
12      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
10      (Teh, Telur)      (Keju)     0.12    0.600000  1.764706
14     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Keju, Telur)       (Teh)     0.12    0.750000  1.630435
11     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
13   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


### Catatan Langkah 4:

Setelah menemukan _frequent itemset_, langkah selanjutnya adalah membentuk _association rules_ (aturan asosiasi) menggunakan fungsi `association_rules` dari `mlxtend`. Aturan asosiasi membantu kita memahami hubungan 'jika ini, maka itu' antar produk. Kita menyaring aturan-aturan ini berdasarkan `min_confidence=0.5` (minimal 50% kemungkinan konsekuen muncul jika anteseden ada) dan `lift > 1` (menunjukkan hubungan positif antara anteseden dan konsekuen).

Output menampilkan 10 aturan asosiasi teratas yang diurutkan berdasarkan nilai _Lift_ tertinggi. _Lift_ adalah metrik penting yang menunjukkan seberapa besar kemungkinan konsekuen muncul ketika anteseden sudah ada, dibandingkan dengan kemungkinan konsekuen muncul secara independen. Nilai _Lift_ lebih dari 1 menunjukkan hubungan positif (produk cenderung dibeli bersama).

Beberapa aturan terkuat yang ditemukan:
- `(Keju, Teh) -> (Telur)` dengan _Lift_ 2.38: Ini adalah aturan yang sangat kuat, menunjukkan bahwa pelanggan yang membeli Keju dan Teh kemungkinan besar juga membeli Telur.
- `(Gula, Roti) -> (Selai)` dengan _Lift_ 1.92: Ini adalah pola yang disuntikkan sebelumnya, dan berhasil ditemukan oleh algoritma, menunjukkan bahwa pelanggan yang membeli Gula dan Roti seringkali juga membeli Selai.
- `(Roti) -> (Selai)` dengan _Lift_ 1.32: Ini adalah salah satu pola utama yang diharapkan, dan meskipun _Lift_-nya tidak setinggi kombinasi lainnya, ini mengkonfirmasi hubungan antara Roti dan Selai.

## Langkah 5: Rekomender Sederhana dengan Content-Based Filtering

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# Bangun katalog produk dengan kategori
katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

# rekomendasi produk serupa menggunakan cosine similarity atas kategori (one-hot)
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


### Catatan Langkah 5:

Pada langkah ini, kita membangun sistem _recommender_ sederhana menggunakan pendekatan _Content-Based Filtering_. Ide utamanya adalah merekomendasikan produk yang memiliki fitur atau atribut serupa dengan produk yang diminati pengguna. Dalam kasus ini, fitur yang digunakan adalah kategori produk.

Kita membuat katalog produk dan menetapkan kategori untuk setiap produk. Kemudian, kita menggunakan _one-hot encoding_ pada kategori untuk membuat representasi fitur dan menghitung _cosine similarity_ antar produk. _Cosine similarity_ mengukur kesamaan arah antara dua vektor fitur, sehingga nilai 1 berarti sangat mirip dan 0 berarti tidak mirip sama sekali.

Fungsi `rekomendasi_serupa` mengambil nama produk dan mengembalikan produk serupa berdasarkan _cosine similarity_ kategori.

Output `Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']` menunjukkan bahwa produk yang dianggap serupa dengan 'Roti' berdasarkan kategori 'Bakery' adalah 'Selai' dan 'Sereal'. 'Susu' juga muncul karena kemungkinan kategori yang sama atau kedekatan dalam representasi fitur.

## Langkah 6: Bandingkan Kedua Pendekatan

In [10]:
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
 lambda x: produk_target in x)]

print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())
print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


### Catatan Langkah 6:

Pada langkah ini, kita membandingkan rekomendasi yang dihasilkan dari dua pendekatan berbeda: *Association Rules* dan *Content-Based Filtering*, dengan contoh produk target 'Roti'.

**1. Apakah kedua pendekatan memberi rekomendasi yang konsisten?**
**Ya, cukup konsisten.** Kedua metode sama-sama merekomendasikan **'Selai'** untuk 'Roti'.
- *Association Rules* menemukannya berdasarkan pola perilaku belanja nyata (orang yang beli Roti cenderung beli Selai).
- *Content-Based* menemukannya berdasarkan kesamaan kategori (keduanya masuk kategori 'Bakery').
Konsistensi ini memberikan tingkat kepercayaan yang tinggi bahwa 'Selai' adalah rekomendasi terbaik untuk pembeli 'Roti'.

**2. Kapan menggunakan salah satu, atau menggabungkan keduanya (Hybrid)?**

*   **Gunakan Association Rules (Market Basket Analysis) saat:**
    - Anda memiliki data transaksi yang banyak.
    - Ingin menemukan hubungan lintas kategori (misal: orang beli Kopi juga beli Mentega) yang tidak terlihat dari fitur produk saja.
    - Fokus pada strategi *cross-selling*.

*   **Gunakan Content-Based Filtering saat:**
    - Ada produk baru yang belum punya riwayat transaksi (*Cold Start Problem*).
    - Anda ingin menyarankan variasi produk dalam kategori yang sama.

*   **Gunakan Pendekatan Hybrid saat:**
    - Anda ingin sistem yang lebih cerdas. Misalnya, gunakan *Content-Based* untuk menyaring kandidat produk serupa, lalu gunakan *Association Rules* untuk mengurutkan mana yang paling mungkin dibeli bersama berdasarkan data historis. Gabungan ini menutupi kelemahan masing-masing metode dan memberikan rekomendasi yang lebih relevan serta personal.

## Kesimpulan Akhir

### 1. Apa yang Dipelajari
Melalui latihan ini, kita telah mempraktikkan dua teknik utama dalam sistem rekomendasi:
*   **Market Basket Analysis (Apriori):** Memahami cara kerja asosiasi antar produk berdasarkan riwayat transaksi nyata.
*   **Content-Based Filtering:** Memahami cara merekomendasikan item berdasarkan kemiripan fitur (kategori) menggunakan *cosine similarity*.

### 2. Temuan Utama
*   **Pola yang Terdeteksi:** Algoritma berhasil mendeteksi pola yang disuntikkan secara manual, yaitu hubungan **Roti -> Selai**.
*   **Aturan Terkuat:** Aturan asosiasi yang memiliki keterikatan paling tinggi adalah **(Keju, Teh) -> (Telur)** dengan nilai Lift sebesar 2.38, menunjukkan adanya potensi paket produk (bundling) di kategori *dairy* dan sarapan.
*   **Validasi Metode:** Kedua metode memberikan hasil yang konsisten untuk produk 'Roti', di mana keduanya menyarankan 'Selai' sebagai item terkait.

### 3. Keterbatasan & Pertanyaan
*   **Ukuran Dataset:** Analisis ini hanya menggunakan 50 transaksi sintetis. Dalam skala industri (ribuan/jutaan transaksi), nilai *support* dan *confidence* mungkin akan jauh lebih rendah namun lebih akurat.
*   **Fitur Terbatas:** Sistem *Content-Based* saat ini hanya menggunakan satu fitur (kategori). Hasil bisa lebih tajam jika menyertakan harga, merek, atau deskripsi teks.
*   **Cold Start:** Muncul pertanyaan mengenai bagaimana menangani user baru yang belum memiliki riwayat transaksi sama sekali, yang bisa dijawab lebih lanjut dengan pendekatan *Collaborative Filtering*.